
CLONE ZA REPO



In [1]:
from pathlib import Path

BASE = Path.cwd()
print("Current directory:", BASE)
print("Contents:")
for p in BASE.iterdir():
    print(" -", p.name)

Current directory: /content
Contents:
 - .config
 - sample_data


In [2]:
import subprocess
from pathlib import Path

REPO = "https://github.com/samiralhaydari/showcase-1.git"

BASE = Path.cwd()
REPO_DIR = BASE / "folder-1"

print("Cloning into:", REPO_DIR)

subprocess.run(
    ["git", "clone", REPO, str(REPO_DIR)],
    check=True,
)

Cloning into: /content/folder-1


CompletedProcess(args=['git', 'clone', 'https://github.com/samiralhaydari/showcase-1.git', '/content/folder-1'], returncode=0)

In [3]:
APP_DIR = REPO_DIR / "mte_components_v0.4"

print(APP_DIR)
print("Exists:", APP_DIR.exists())

%cd {APP_DIR}

/content/folder-1/mte_components_v0.4
Exists: True
/content/folder-1/mte_components_v0.4


INSTALL

In [4]:
!pip install -q gradio

In [5]:
!curl https://elan.lean-lang.org/elan-init.sh -sSf | sh -s -- -y

info: downloading installer
info: default toolchain set to 'stable'


In [6]:
import os

os.environ["PATH"] = (
    os.path.expanduser("~/.elan/bin")
    + ":"
    + os.environ["PATH"]
)

In [7]:
!lean --version
!lake --version

info: downloading https://releases.lean-lang.org/lean4/v4.35.0/lean-4.35.0-linux.tar.zst
566.9 MiB / 566.9 MiB (100 %) 133.7 MiB/s ETA:   0 s
info: installing /root/.elan/toolchains/leanprover--lean4---v4.35.0
Lean (version 4.35.0, x86_64-unknown-linux-gnu, commit eb6cafb7290e8bd5f2937522d8d804f58f02bafc, Release)
Lake version 5.0.0-src+eb6cafb (Lean version 4.35.0)


In [8]:
from pathlib import Path
import os

BASE = Path.cwd()
LEAN_DIR = BASE / "mte_lean"

print("Lean project will be created at:", LEAN_DIR)

Lean project will be created at: /content/folder-1/mte_components_v0.4/mte_lean


In [9]:
%cd {BASE}

!lake new mte_lean math

/content/folder-1/mte_components_v0.4
info: mte_lean: no previous manifest, creating one from scratch
info: leanprover-community/mathlib: materializing new dependency
info: leanprover-community/mathlib: fetching revision 'v4.35.0' from https://github.com/leanprover-community/mathlib4
info: leanprover-community/mathlib: checking out revision '56a005855ea17e0b0958f73ec801ae140b0a1679'
info: plausible: materializing new dependency
info: plausible: fetching revision 'dfa4f38190d07c37e315953e08b90fe2d95b3c24' from https://github.com/leanprover-community/plausible
info: plausible: checking out revision 'dfa4f38190d07c37e315953e08b90fe2d95b3c24'
info: LeanSearchClient: materializing new dependency
info: LeanSearchClient: fetching revision 'd6b4c7be5be75ec4917c873ac50c756dc1436372' from https://github.com/leanprover-community/LeanSearchClient
info: LeanSearchClient: checking out revision 'd6b4c7be5be75ec4917c873ac50c756dc1436372'
info: importGraph: materializing new dependency
info: importGrap

In [10]:
%cd {LEAN_DIR}

!lake update

/content/folder-1/mte_components_v0.4/mte_lean
info: leanprover-community/mathlib: fetching revision 'v4.35.0' from https://github.com/leanprover-community/mathlib4
info: toolchain not updated; already up-to-date
info: mathlib: running post-update hooks
Current branch: HEAD
Using cache from origin: (some leanprover-community/mathlib4)
No files to download
Already decompressed 9031 file(s)


In [11]:
!lake exe cache get

Current branch: HEAD
Using cache from origin: (some leanprover-community/mathlib4)
No files to download
Already decompressed 9031 file(s)


test lean on python

In [12]:
from pathlib import Path
import subprocess

# ---------------------------------------------------------
# 1. Try the expected Lean project location
# ---------------------------------------------------------

BASE = Path.cwd()
LEAN_DIR = BASE / "mte_lean"

# ---------------------------------------------------------
# 2. If not found there, search for an existing Lake project
# ---------------------------------------------------------

def find_lean_project():
    expected = BASE / "mte_lean"

    if expected.exists():
        return expected

    lakefiles = (
        list(BASE.rglob("lakefile.toml"))
        + list(BASE.rglob("lakefile.lean"))
    )

    if not lakefiles:
        return None

    # Prefer a folder actually named mte_lean if available
    for lakefile in lakefiles:
        if lakefile.parent.name == "mte_lean":
            return lakefile.parent

    # Otherwise use the first Lean/Lake project found
    return lakefiles[0].parent


LEAN_DIR = find_lean_project()

if LEAN_DIR is None:
    raise FileNotFoundError(
        "No Lean project was found.\n"
        "Create one first with:\n\n"
        "    !lake new mte_lean math\n"
    )

LEAN_PROJECT = str(LEAN_DIR)

print("LEAN_PROJECT =", LEAN_PROJECT)
print("Exists:", LEAN_DIR.exists())

# ---------------------------------------------------------
# 3. Check that Lake is available
# ---------------------------------------------------------

lake_check = subprocess.run(
    ["lake", "--version"],
    text=True,
    capture_output=True,
)

if lake_check.returncode != 0:
    raise RuntimeError(
        "Lake is not available on PATH.\n"
        "Make sure Lean/elán is installed and ~/.elan/bin is on PATH."
    )

print("Lake:", lake_check.stdout.strip())

# ---------------------------------------------------------
# 4. Create a Lean test file
# ---------------------------------------------------------

test_file = LEAN_DIR / "Test.lean"

test_file.write_text(
    """import Mathlib

example (x : ℝ) : 0 ≤ x^2 := by
  positivity
""",
    encoding="utf-8",
)

print("Created:", test_file)

# ---------------------------------------------------------
# 5. Run Lean through Lake
# ---------------------------------------------------------

result = subprocess.run(
    ["lake", "env", "lean", "Test.lean"],
    cwd=LEAN_PROJECT,
    text=True,
    capture_output=True,
)

print("\nReturn code:", result.returncode)

if result.stdout:
    print("\nSTDOUT:")
    print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

# ---------------------------------------------------------
# 6. Final status
# ---------------------------------------------------------

if result.returncode == 0:
    print("\n✓ Lean + Mathlib are working correctly.")
else:
    print("\n✗ Lean test failed.")

LEAN_PROJECT = /content/folder-1/mte_components_v0.4/mte_lean
Exists: True
Lake: Lake version 5.0.0-src+eb6cafb (Lean version 4.35.0)
Created: /content/folder-1/mte_components_v0.4/mte_lean/Test.lean

Return code: 0

✓ Lean + Mathlib are working correctly.


# connect the notebook to v0.4 code

In [13]:
from pathlib import Path
import subprocess
import sys
import importlib

# ---------------------------------------------------------
# 1. Resolve Git repository root
# ---------------------------------------------------------

HERE = Path.cwd().resolve()

result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    cwd=HERE,
    text=True,
    capture_output=True,
    check=True,
)

GIT_ROOT = Path(result.stdout.strip()).resolve()

print("Current directory :", HERE)
print("Git root          :", GIT_ROOT)

# ---------------------------------------------------------
# 2. Select v0.4 EXPLICITLY
# ---------------------------------------------------------

APP_DIR = GIT_ROOT / "mte_components_v0.4"

print("APP_DIR           :", APP_DIR)

if not APP_DIR.exists():
    raise FileNotFoundError(
        f"Expected v0.4 directory does not exist:\n{APP_DIR}"
    )

required_files = [
    APP_DIR / "mte_app" / "__init__.py",
    APP_DIR / "mte_app" / "lean_diagnostics.py",
    APP_DIR / "mte_app" / "pedagogical.py",
]

for file in required_files:
    print(
        "✓" if file.exists() else "✗",
        file
    )

missing = [p for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "v0.4 is missing required files:\n"
        + "\n".join(str(p) for p in missing)
    )

# ---------------------------------------------------------
# 3. Remove any previously cached v0.2/v0.3 mte_app modules
# ---------------------------------------------------------

for module_name in list(sys.modules):
    if (
        module_name == "mte_app"
        or module_name.startswith("mte_app.")
    ):
        del sys.modules[module_name]

# Remove older MTE component paths from sys.path
sys.path = [
    p for p in sys.path
    if "mte_components_v0.3" not in p
    and "mte_components-v0.2" not in p
    and "mte_components_v0.2" not in p
]

# Put v0.4 FIRST
sys.path.insert(0, str(APP_DIR))

importlib.invalidate_caches()

# ---------------------------------------------------------
# 4. Import specifically from v0.4
# ---------------------------------------------------------

from mte_app.lean_diagnostics import parse_lean_diagnostic
from mte_app.pedagogical import generate_socratic_hint

# ---------------------------------------------------------
# 5. Confirm exactly which files Python imported
# ---------------------------------------------------------

import mte_app
import mte_app.lean_diagnostics
import mte_app.pedagogical

print("\nImported mte_app from:")
print(Path(mte_app.__file__).resolve())

print("\nImported lean_diagnostics from:")
print(Path(mte_app.lean_diagnostics.__file__).resolve())

print("\nImported pedagogical from:")
print(Path(mte_app.pedagogical.__file__).resolve())

print("\n✓ MTE v0.4 imported correctly.")

Current directory : /content/folder-1/mte_components_v0.4/mte_lean
Git root          : /content/folder-1
APP_DIR           : /content/folder-1/mte_components_v0.4
✓ /content/folder-1/mte_components_v0.4/mte_app/__init__.py
✓ /content/folder-1/mte_components_v0.4/mte_app/lean_diagnostics.py
✓ /content/folder-1/mte_components_v0.4/mte_app/pedagogical.py

Imported mte_app from:
/content/folder-1/mte_components_v0.4/mte_app/__init__.py

Imported lean_diagnostics from:
/content/folder-1/mte_components_v0.4/mte_app/lean_diagnostics.py

Imported pedagogical from:
/content/folder-1/mte_components_v0.4/mte_app/pedagogical.py

✓ MTE v0.4 imported correctly.


In [14]:
import subprocess
import time
from pathlib import Path

LEAN_DIR = Path(LEAN_PROJECT).resolve()

print("Preparing Lean/Mathlib project at:")
print(LEAN_DIR)

print("\n[1/3] Updating dependencies...")
result = subprocess.run(
    ["lake", "update"],
    cwd=LEAN_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("lake update failed")


print("\n[2/3] Downloading Mathlib cache...")
result = subprocess.run(
    ["lake", "exe", "cache", "get"],
    cwd=LEAN_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Mathlib cache download failed")


print("\n[3/3] Warming up Lean...")

warmup_file = LEAN_DIR / "Warmup.lean"

warmup_file.write_text(
    """import Mathlib

example : True := by
  trivial
""",
    encoding="utf-8",
)

start = time.perf_counter()

result = subprocess.run(
    ["lake", "env", "lean", "Warmup.lean"],
    cwd=LEAN_DIR,
    text=True,
    capture_output=True,
    timeout=300,
)

elapsed = time.perf_counter() - start

print("Warmup time:", round(elapsed, 2), "seconds")
print("Return code:", result.returncode)

if result.stdout:
    print(result.stdout)

if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("\n✓ Lean + Mathlib warmed up successfully.")
else:
    print("\n✗ Warmup failed.")

Preparing Lean/Mathlib project at:
/content/folder-1/mte_components_v0.4/mte_lean

[1/3] Updating dependencies...
Current branch: HEAD
Using cache from origin: (some leanprover-community/mathlib4)
No files to download
Already decompressed 9031 file(s)

info: leanprover-community/mathlib: fetching revision 'v4.35.0' from https://github.com/leanprover-community/mathlib4
info: toolchain not updated; already up-to-date
info: mathlib: running post-update hooks


[2/3] Downloading Mathlib cache...
Current branch: HEAD
Using cache from origin: (some leanprover-community/mathlib4)
No files to download
Already decompressed 9031 file(s)



[3/3] Warming up Lean...
Warmup time: 28.55 seconds
Return code: 0

✓ Lean + Mathlib warmed up successfully.


In [15]:
from pathlib import Path
import subprocess
import tempfile

# ---------------------------------------------------------
# Lean project
# ---------------------------------------------------------

LEAN_DIR = Path(LEAN_PROJECT).resolve()

print("Using Lean project:", LEAN_DIR)

# Sanity checks
if not LEAN_DIR.exists():
    raise FileNotFoundError(f"Lean project does not exist: {LEAN_DIR}")

if not (
    (LEAN_DIR / "lakefile.toml").exists()
    or (LEAN_DIR / "lakefile.lean").exists()
):
    raise RuntimeError(
        f"{LEAN_DIR} does not appear to be a Lake project."
    )


# ---------------------------------------------------------
# Helper: Socratic feedback
# ---------------------------------------------------------

def lean_hint(raw_output, diagnostic):
    """
    Use the v0.4 pedagogical hint generator if possible.
    Falls back to a simple Socratic question if its signature differs.
    """

    try:
        return generate_socratic_hint(
            "Lean",
            raw_output,
            diagnostic.summary,
        )

    except TypeError:
        # Defensive fallback in case your v0.4 function uses
        # a slightly different signature.
        try:
            return generate_socratic_hint(
                "Lean",
                raw_output,
            )
        except Exception:
            pass

    except Exception:
        pass

    # Fallback Socratic hints
    text = raw_output.lower()

    if "unsolved goals" in text:
        return (
            "Which goal is still open, and what mathematical fact "
            "would directly justify it?"
        )

    if "unknown identifier" in text:
        return (
            "Which identifier is Lean unable to resolve? "
            "Is it defined, imported, or spelled correctly?"
        )

    if "type mismatch" in text:
        return (
            "What type does Lean expect here, and what type does "
            "your expression actually have?"
        )

    if "tactic" in text and "failed" in text:
        return (
            "What assumptions are available at this point, and does "
            "the chosen tactic apply to this kind of goal?"
        )

    return (
        "What is the current goal, and which assumption or theorem "
        "could move you one step closer to proving it?"
    )


# ---------------------------------------------------------
# Main Lean execution function
# ---------------------------------------------------------

def run_lean(code: str):
    """
    Compile/verify Lean code inside the configured Mathlib project.

    Returns:
        raw_output
        verification_status
        socratic_hint
    """

    code = (code or "").strip()

    if not code:
        return (
            "",
            "⚠ No Lean code submitted.",
            "What theorem or conjecture would you like to formalise?",
        )

    source_file = None

    try:
        # Create temporary Lean source INSIDE the Lake project
        with tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".lean",
            prefix="mte_student_",
            dir=LEAN_DIR,
            delete=False,
            encoding="utf-8",
        ) as f:
            f.write(code)
            source_file = Path(f.name)

        # Run Lean using the Lake environment
        result = subprocess.run(
            [
                "lake",
                "env",
                "lean",
                source_file.name,
            ],
            cwd=LEAN_DIR,
            text=True,
            capture_output=True,
            timeout=120,
        )

        # Collect stdout + stderr
        chunks = []

        if result.stdout.strip():
            chunks.append(result.stdout.strip())

        if result.stderr.strip():
            chunks.append(result.stderr.strip())

        raw_output = "\n\n".join(chunks)

        # Parse using your MTE v0.4 diagnostic layer
        diagnostic = parse_lean_diagnostic(
            raw_output,
            result.returncode,
        )

        # -------------------------------------------------
        # Verification display
        # -------------------------------------------------

        status = [diagnostic.title]

        if diagnostic.summary:
            status.append(diagnostic.summary)

        if diagnostic.line is not None:
            location = f"Line {diagnostic.line}"

            if diagnostic.column is not None:
                location += f", column {diagnostic.column}"

            status.append(location)

        if diagnostic.verified:
            if diagnostic.severity == "warning":
                status.append(
                    "Lean accepted the proof with a warning."
                )
            else:
                status.append(
                    "Machine-checked by Lean."
                )

        verification_status = "\n".join(status)

        # -------------------------------------------------
        # Pedagogical feedback
        # -------------------------------------------------

        if diagnostic.verified:
            hint = (
                "Lean accepted the proof. Now check the formalisation: "
                "does this theorem state exactly the mathematical claim "
                "you intended to prove?"
            )
        else:
            hint = lean_hint(
                raw_output,
                diagnostic,
            )

        return (
            raw_output or "(Lean produced no console output.)",
            verification_status,
            hint,
        )

    except subprocess.TimeoutExpired:
        return (
            "Lean verification exceeded the 120-second limit",
            "⚠ VERIFICATION TIMED OUT",
            (
                "Could the proof strategy be simplified, or is a tactic "
                "search exploring too large a space?"
            ),
        )

    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"

        return (
            message,
            "✗ LEAN EXECUTION ERROR",
            (
                "Check whether the Lake project is valid and whether "
                "`lake env lean Test.lean` works from this directory."
            ),
        )

    finally:
        if source_file is not None:
            try:
                source_file.unlink(missing_ok=True)
            except Exception:
                pass

Using Lean project: /content/folder-1/mte_components_v0.4/mte_lean


In [16]:
GOOD_LEAN = """
import Mathlib

theorem square_nonnegative (x : ℝ) :
    0 ≤ x^2 := by
  positivity
"""

raw, status, hint = run_lean(GOOD_LEAN)

print("===== STATUS =====")
print(status)

print("\n===== RAW OUTPUT =====")
print(raw)

print("\n===== SOCRATIC FEEDBACK =====")
print(hint)

===== STATUS =====
✓ PROOF VERIFIED
Lean accepted the submission. No proof errors were reported.
Machine-checked by Lean.

===== RAW OUTPUT =====
(Lean produced no console output.)

===== SOCRATIC FEEDBACK =====
Lean accepted the proof. Now check the formalisation: does this theorem state exactly the mathematical claim you intended to prove?


In [17]:
BAD_LEAN = """
import Mathlib

theorem impossible_claim (x : ℝ) :
    x > x := by
  linarith
"""

raw, status, hint = run_lean(BAD_LEAN)

print("===== STATUS =====")
print(status)

print("\n===== RAW OUTPUT =====")
print(raw)

print("\n===== SOCRATIC FEEDBACK =====")
print(hint)

===== STATUS =====
✗ PROOF REJECTED
linarith failed to find a contradiction
Line 5, column 2

===== RAW OUTPUT =====
mte_student_j0em233m.lean:5:2: error: linarith failed to find a contradiction
x : ℝ
a✝ : x ≤ x
⊢ False
failed

===== SOCRATIC FEEDBACK =====
Find the first Lean message that changes the proof state. What does Lean say it has, what does it still need, and which assumption or lemma could bridge that gap?


# Gradio UI code goes here

# Wire za snek

In [18]:
import subprocess
import tempfile
import sys
from pathlib import Path


def python_hint(raw_output: str):
    """
    Use the MTE v0.4 Socratic hint generator where possible.
    Fall back to question-based debugging prompts.
    """

    try:
        return generate_socratic_hint(
            "Python",
            raw_output,
            raw_output,
        )
    except TypeError:
        try:
            return generate_socratic_hint(
                "Python",
                raw_output,
            )
        except Exception:
            pass
    except Exception:
        pass

    text = raw_output.lower()

    if "nameerror" in text:
        return (
            "Which variable or function is Python unable to find? "
            "Where should it have been defined before this line?"
        )

    if "typeerror" in text:
        return (
            "What types are involved in the failing operation? "
            "Are those types compatible with the operation you intended?"
        )

    if "indexerror" in text:
        return (
            "What is the size of the sequence, and which index is your "
            "program attempting to access?"
        )

    if "zerodivisionerror" in text:
        return (
            "Under what input conditions does the denominator become zero? "
            "Should those conditions be excluded or handled explicitly?"
        )

    if "syntaxerror" in text:
        return (
            "Which part of the statement does Python identify as invalid? "
            "Check brackets, colons, indentation, and expression structure."
        )

    if "modulenotfounderror" in text:
        return (
            "Which package is missing? Is it part of the standard library, "
            "or does it need to be installed in this runtime?"
        )

    return (
        "What result did you expect at the point of failure, and what value "
        "or state did the program actually produce?"
    )


def run_python(code: str):
    """
    Execute student Python code in a separate subprocess.

    Returns:
        python_output
        socratic_hint
    """

    code = (code or "").strip()

    if not code:
        return (
            "⚠ No Python code submitted.",
            "What computational experiment would help test your conjecture?"
        )

    source_file = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".py",
            prefix="mte_student_",
            delete=False,
            encoding="utf-8",
        ) as f:
            f.write(code)
            source_file = Path(f.name)

        result = subprocess.run(
            [
                sys.executable,
                "-I",
                "-u",
                str(source_file),
            ],
            text=True,
            capture_output=True,
            timeout=30,
        )

        chunks = []

        if result.stdout.strip():
            chunks.append(result.stdout.strip())

        if result.stderr.strip():
            chunks.append(result.stderr.strip())

        raw_output = "\n\n".join(chunks)

        if result.returncode == 0:

            if not raw_output:
                raw_output = "✓ Python completed successfully with no output."

            hint = (
                "The Python experiment completed successfully. "
                "What does this computational evidence suggest about the "
                "conjecture, and what does it still not prove?"
            )

        else:

            if not raw_output:
                raw_output = (
                    f"Python exited with code {result.returncode}."
                )

            hint = python_hint(raw_output)

        return raw_output, hint

    except subprocess.TimeoutExpired:
        return (
            "⚠ Python execution exceeded the 30-second limit.",
            (
                "Is there an infinite loop, an excessively large computation, "
                "or a condition that never becomes false?"
            ),
        )

    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"

        return (
            message,
            (
                "Is the failure caused by the student's Python program, "
                "or by the execution environment itself?"
            ),
        )

    finally:
        if source_file is not None:
            try:
                source_file.unlink(missing_ok=True)
            except Exception:
                pass

# ZA MATLAB

In [19]:
!apt-get -qq update
!apt-get -qq install -y octave gnuplot-nox

!octave --version | head -3

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 122806 files and directories currently installed.)
Preparing to unpack .../0-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package liblqr-1-0:amd64.
Preparing to unpack .../1-liblqr-1-0_0.4.2-2.1build2_amd64.deb ...
Unpacking liblqr-1-0:amd64 (0.4.2-2.1build2) ...
Selecting previously unselected package libraw23t64:amd64.
Preparing to unpack .../2-libraw23t64_0.21.2-2.1ubuntu0.24.04.2_amd64.deb ...
Unpacking libraw23t64:amd64 (0.21.2-2.1ubuntu0.24.04.2) ...
Selecting previously unselected package imagemagick-6-common.
Preparing to unpack .../3-imagemagick-6-common_8%3a6.9.12.98+d

the full matlab thing

In [20]:
import subprocess
import tempfile
import shutil
from pathlib import Path


def matlab_hint(raw_output: str):
    """
    Socratic feedback for MATLAB/Octave execution.
    """

    try:
        return generate_socratic_hint(
            "MATLAB",
            raw_output,
            raw_output,
        )
    except TypeError:
        try:
            return generate_socratic_hint(
                "MATLAB",
                raw_output,
            )
        except Exception:
            pass
    except Exception:
        pass

    text = raw_output.lower()

    if "undefined" in text:
        return (
            "Which variable or function is undefined? "
            "Where should it have been introduced before this line?"
        )

    if "dimension" in text or "nonconformant" in text:
        return (
            "What are the dimensions of the arrays involved? "
            "Are the matrix operations dimensionally compatible?"
        )

    if "index" in text:
        return (
            "What is the size of the array, and which index is being accessed?"
        )

    if "parse error" in text or "syntax error" in text:
        return (
            "Which expression does the interpreter reject? "
            "Check brackets, operators, commas, and statement endings."
        )

    return (
        "What numerical or graphical result did you expect, "
        "and what did the computation actually produce?"
    )


# ---------------------------------------------------------
# PUT THIS HERE — ABOVE run_matlab()
# ---------------------------------------------------------

def clean_octave_output(text: str) -> str:
    """
    Remove Octave headless plotting noise:
    1. gnuplot warning paragraph
    2. ASCII terminal plot blocks
    """

    lines = text.splitlines()
    cleaned = []

    # ---------------------------------------------
    # 1. Remove gnuplot warning paragraph
    # ---------------------------------------------
    skipping_gnuplot_warning = False

    # ---------------------------------------------
    # 2. Remove ASCII plot block
    # ---------------------------------------------
    skipping_ascii_plot = False

    for line in lines:
        lower = line.lower()
        stripped = line.rstrip()

        # -----------------------------------------
        # Start of gnuplot warning block
        # -----------------------------------------
        if "warning: using the gnuplot graphics toolkit" in lower:
            skipping_gnuplot_warning = True
            continue

        if skipping_gnuplot_warning:
            if "the qt toolkit is recommended instead" in lower:
                skipping_gnuplot_warning = False
            continue

        # -----------------------------------------
        # Detect start of ASCII plot
        # Common signs:
        #   |-----title-----|
        #   +----...----+
        # -----------------------------------------
        if (
            ("|---" in stripped or "|--" in stripped)
            and stripped.strip().startswith("|")
        ):
            skipping_ascii_plot = True
            continue

        # While inside ASCII plot, skip everything
        if skipping_ascii_plot:
            # A rough end condition:
            # line with x-axis labels usually ends the block
            if stripped.strip() == "":
                skipping_ascii_plot = False
            continue

        cleaned.append(line)

    # Final cleanup: collapse extra blank lines
    result_lines = []
    blank = False

    for line in cleaned:
        if line.strip() == "":
            if not blank:
                result_lines.append("")
            blank = True
        else:
            result_lines.append(line)
            blank = False

    return "\n".join(result_lines).strip()


# ---------------------------------------------------------
# MATLAB runner
# ---------------------------------------------------------

def run_matlab(code: str):
    """
    MATLAB-compatible execution using GNU Octave on hosted Colab.

    Returns:
        console_output
        socratic_hint
        generated_figures
    """

    code = (code or "").strip()

    if not code:
        return (
            "⚠ No MATLAB-compatible code submitted.",
            "What computational experiment would help explore the problem?",
            [],
        )

    if shutil.which("octave") is None:
        return (
            "GNU Octave is not installed.",
            "Run the Octave installation cell first.",
            [],
        )

    work_dir = Path(
        tempfile.mkdtemp(prefix="mte_octave_")
    )

    script_file = work_dir / "student_submission.m"

    headless_setup = r"""
% ---------- MTE headless plotting setup ----------
warning("off", "all");
set(0, "defaultfigurevisible", "off");
"""

    figure_capture = r"""
% ---------- MTE automatic figure capture ----------
try
    __mte_figs__ = findall(0, "type", "figure");

    for __mte_i__ = 1:numel(__mte_figs__)
        figure(__mte_figs__(__mte_i__));

        __mte_name__ = sprintf(
            "__mte_figure_%02d.png",
            __mte_i__
        );

        print(
            __mte_name__,
            "-dpng",
            "-r120"
        );
    endfor

    clear __mte_figs__ __mte_i__ __mte_name__;
catch
    % Figure capture failure should not invalidate numerical work.
end_try_catch
"""

    script_file.write_text(
        headless_setup
        + "\n"
        + code
        + "\n"
        + figure_capture,
        encoding="utf-8",
    )

    try:
        result = subprocess.run(
            [
                "octave",
                "--quiet",
                "--no-gui",
                str(script_file),
            ],
            cwd=work_dir,
            text=True,
            capture_output=True,
            timeout=60,
        )

        chunks = []

        if result.stdout.strip():
            chunks.append(result.stdout.strip())

        if result.stderr.strip():
            chunks.append(result.stderr.strip())

        raw_output = clean_octave_output(
            "\n\n".join(chunks)
        )

        figures = [
            str(path)
            for path in sorted(
                work_dir.glob("__mte_figure_*.png")
            )
        ]

        if result.returncode == 0:

            if not raw_output:
                raw_output = (
                    "✓ MATLAB-compatible computation completed successfully."
                )

            hint = (
                "The numerical experiment completed successfully. "
                "What pattern does this evidence suggest, and what "
                "would still require mathematical justification?"
            )

        else:

            if not raw_output:
                raw_output = (
                    f"Octave exited with code {result.returncode}."
                )

            hint = matlab_hint(raw_output)

        return (
            raw_output,
            hint,
            figures,
        )

    except subprocess.TimeoutExpired:
        return (
            "⚠ MATLAB-compatible execution exceeded the 60-second limit.",
            (
                "Is the computation unnecessarily large, "
                "or is there a loop whose stopping condition is not reached?"
            ),
            [],
        )

    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"

        return (
            message,
            (
                "Is this an error in the submitted computation, "
                "or in the execution environment?"
            ),
            [],
        )

# dis is the gradio bro

In [21]:
import gradio as gr


LEAN_STARTER = """import Mathlib

theorem square_nonnegative (x : ℝ) :
    0 ≤ x^2 := by
  positivity
"""


with gr.Blocks(title="Mathematical Thinking Engine") as mte_app:

    gr.Markdown(
        """
# Mathematical Thinking Engine

**MATLAB exploration → Python conjecture testing → Lean formal verification**
"""
    )

    # -----------------------------------------------------
    # Shared problem / conjecture
    # -----------------------------------------------------

    problem = gr.Textbox(
        label="Problem / Conjecture",
        value=(
            "Prove that the square of every real number "
            "is non-negative."
        ),
        lines=3,
    )

    gr.Markdown("---")

    # -----------------------------------------------------
    # Three-column workspace
    # -----------------------------------------------------

    with gr.Row():

        # =================================================
        # MATLAB
        # =================================================

        with gr.Column():

            gr.Markdown("## MATLAB")

            gr.Markdown(
                """
        **Hosted Colab mode:** MATLAB-compatible execution using GNU Octave.

        Numerical results can be used as computational evidence,
        but this runtime is not MathWorks MATLAB.
        """
            )

            matlab_code = gr.Textbox(
                label="MATLAB Code",
                lines=16,
                value="""x = linspace(-5, 5, 100);
        y = x.^2;

        disp("Minimum sampled value:");
        disp(min(y));

        plot(x, y);
        grid on;
        title("y = x^2");
        xlabel("x");
        ylabel("y");
        """,
            )

            matlab_run = gr.Button(
                "▶ Run MATLAB-compatible",
            )

            matlab_output = gr.Textbox(
                label="MATLAB-compatible Output",
                lines=10,
                interactive=False,
            )

            matlab_hint = gr.Textbox(
                label="Socratic Hint",
                lines=4,
                interactive=False,
            )

            matlab_figures = gr.Gallery(
                label="Generated Figures",
                columns=1,
                height=300,
            )

        # =================================================
        # PYTHON
        # =================================================

        with gr.Column():

            gr.Markdown("## Python")

            python_code = gr.Textbox(
                label="Python Code",
                lines=16,
                placeholder="Python code will go here...",
            )

            python_run = gr.Button(
                "▶ Run Python",
            )

            python_output = gr.Textbox(
                label="Python Output",
                lines=10,
                interactive=False,
            )

            python_hint = gr.Textbox(
                label="Socratic Hint",
                lines=4,
                interactive=False,
            )

        # =================================================
        # LEAN 4
        # =================================================

        with gr.Column():

            gr.Markdown("## Lean 4")

            lean_code = gr.Textbox(
                label="Lean Code",
                value=LEAN_STARTER,
                lines=16,
            )

            lean_run = gr.Button(
                "✓ Verify Lean",
                variant="primary",
            )

            lean_status = gr.Textbox(
                label="Formal Verification",
                lines=4,
                interactive=False,
            )

            lean_output = gr.Textbox(
                label="Lean Output",
                lines=10,
                interactive=False,
            )

            lean_hint = gr.Textbox(
                label="Socratic Hint",
                lines=5,
                interactive=False,
            )

    # -----------------------------------------------------
    # MATLAB-compatible wiring
    # -----------------------------------------------------

    matlab_run.click(
        fn=run_matlab,
        inputs=matlab_code,
        outputs=[
            matlab_output,
            matlab_hint,
            matlab_figures,
        ],
    )


    # -----------------------------------------------------
    # Python wiring
    # -----------------------------------------------------

    python_run.click(
        fn=run_python,
        inputs=python_code,
        outputs=[
            python_output,
            python_hint,
        ],
    )


    # -----------------------------------------------------
    # Lean wiring
    # -----------------------------------------------------

    lean_run.click(
        fn=run_lean,
        inputs=lean_code,
        outputs=[
            lean_output,
            lean_status,
            lean_hint,
        ],
    )

# **LAUNCH ZA APP**

In [ ]:
mte_app.launch(
    share=True,
    debug=True,
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bb9ff9404a3a30f44a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
